In [66]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import os
from collections import defaultdict
from scipy.signal import find_peaks
from scipy.interpolate import interp1d

### Импорт данных

In [67]:
def load_motion_csv(filepath):
    """Парсит CSV в формате BVH/MOTION."""
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        lines = [line.strip() for line in f.readlines()]

    header_idx = None
    for i, line in enumerate(lines):
        if "lhip-fle-ext" in line or "head-fle-ext" in line:
            header_idx = i
            break
    if header_idx is None:
        return None

    col_names = [c.strip() for c in lines[header_idx].split(",")]
    data_rows = []
    for line in lines[header_idx + 1:]:
        if not line or line.startswith(("MOTION", "Frames", "Frame Time")):
            continue
        parts = [p.strip() for p in line.split(",")]
        if len(parts) != len(col_names):
            continue
        try:
            data_rows.append([float(x) for x in parts])
        except ValueError:
            continue

    if not data_rows:
        return None
    return pd.DataFrame(data_rows, columns=col_names)


def load_patients_from_dir(base_dir):
    """Загружает все CSV пациентов из папки."""
    patients_data = defaultdict(list)
    bad_files = []

    for filename in sorted(os.listdir(base_dir)):
        if not filename.lower().endswith(".csv"):
            continue
        patient_name = filename.split("_")[0].rsplit(" ", 1)[0]
        full_path = os.path.join(base_dir, filename)

        try:
            df = load_motion_csv(full_path)
            if df is None or df.empty:
                raise ValueError("Не удалось извлечь блок MOTION")
            patients_data[patient_name].append(df)
        except Exception as e:
            print(f"Ошибка в файле: {filename}: {e}")
            bad_files.append(filename)

    return dict(patients_data), bad_files


In [68]:
base_dir_tta = r"C:\Users\Dexter\Desktop\May-databases\17_june\PN3"
patients_data_tta, bad_files_tta = load_patients_from_dir(base_dir_tta)
print("TTA. Пациенты:", list(patients_data_tta.keys()))
print("Файлы с ошибками:", bad_files_tta)

TTA. Пациенты: ['Абаимов Алексей Николаевич (протез C-Leg)', 'Абаимов Алексей Николаевич (протез Спутник)', 'Абаимов Алексей Николаевич (протезист Вайцель протез Symphony)', 'Абаимов Алексей Николаевич (протезист Яргин протез Symphony)', 'Терешков Дмитрий Александрович (протез Orion3)', 'Терешков Дмитрий Александрович (протезист Сиденко протез Спутник)', 'Терешков Дмитрий Александрович (протезист Сугатов протез Symphony)']
Файлы с ошибками: []


In [69]:
LIMB_COLS = [
    "lhip-fle-ext", "lhip-med-lat", "lhip-add-abd",
    "rhip-fle-ext", "rhip-med-lat", "rhip-add-abd",
    "lknee-fle-ext", "lknee-med-lat", "lknee-add-abd",
    "rknee-fle-ext", "rknee-med-lat", "rknee-add-abd",
    "lankle-fle-ext", "lankle-med-lat", "lankle-inv-eve",
    "rankle-fle-ext", "rankle-med-lat", "rankle-inv-eve",
]

for patient_name, df_list in patients_data_tta.items():
    for df in df_list:
        for col in ("lankle-fle-ext", "rankle-fle-ext"):
            if col in df.columns:
                df[col] = -df[col].astype(float)
        for col in LIMB_COLS:
            if col in df.columns:
                df[col] = df[col].astype(float) - df[col].astype(float)[0]

MOTION_CHANNELS = {
    "hip": ["lhip-fle-ext", "lhip-med-lat", "lhip-add-abd", "rhip-fle-ext", "rhip-med-lat", "rhip-add-abd"],
    "knee": ["lknee-fle-ext", "lknee-med-lat", "lknee-add-abd", "rknee-fle-ext", "rknee-med-lat", "rknee-add-abd"],
    "ankle": ["lankle-fle-ext", "lankle-med-lat", "lankle-inv-eve", "rankle-fle-ext", "rankle-med-lat", "rankle-inv-eve"],
    "truncus": ["truncus-fle-ext", "truncus-rot", "truncus-lat"],
    "com": ["COM-X", "COM-Y", "COM-Z"],
}

FLE_EXT_COLS = ["lhip-fle-ext", "rhip-fle-ext", "lknee-fle-ext", "rknee-fle-ext", "lankle-fle-ext", "rankle-fle-ext"]

LIMB_COLUMNS = {
    "L Hip": ["lhip-fle-ext"],
    "R Hip": ["rhip-fle-ext"],
    "L Knee": ["lknee-fle-ext"],
    "R Knee": ["rknee-fle-ext"],
    "L Ankle": ["lankle-fle-ext"],
    "R Ankle": ["rankle-fle-ext"],
}

FRAME_RATE = 60
CM_TO_M = 100



### Визуализация данных


In [97]:
def plot_limbs_plotly(df, title=None, frame_time=1.0/60):
    """Строит интерактивные графики Plotly по данным конечностей."""
    n_limbs = len(LIMB_COLUMNS)
    fig = make_subplots(rows=n_limbs, cols=1, subplot_titles=list(LIMB_COLUMNS.keys()),
                        vertical_spacing=0.06, shared_xaxes=True)
    n_frames = len(df)
    # t = np.arange(n_frames) * frame_time
    t = np.arange(n_frames)
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

    for row, (limb_name, cols) in enumerate(LIMB_COLUMNS.items(), start=1):
        for i, col in enumerate(cols):
            if col not in df.columns:
                continue
            y = df[col].astype(float).values.copy()
            fig.add_trace(go.Scatter(x=t, y=y, name=col, line=dict(width=1.5, color=colors[i % 3])), row=row, col=1)

    fig.update_layout(height=180*n_limbs, title_text=title or "Углы суставов", showlegend=True,
                      legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1), margin=dict(t=80, b=40))
    fig.update_xaxes(title_text="Время, с", row=n_limbs, col=1)
    fig.update_yaxes(title_text="град.", col=1)
    fig.show()
    return fig

idx=0
if patients_data_tta:
    first_patient_tta = list(patients_data_tta.keys())[6]
    plot_limbs_plotly(patients_data_tta[first_patient_tta][idx], title=f"Пациент: {first_patient_tta}, файл {idx+1}")

    


### Алгоритмы для выделения циклов походки


In [71]:

def interpolate_cycle(cycle, target_size=100):
    """Приводит цикл к фиксированной длине через интерполяцию."""
    cycle = np.asarray(cycle, dtype=float)
    if len(cycle) == 0:
        return np.full(target_size, np.nan)
    if len(cycle) == 1:
        return np.full(target_size, cycle[0])
    x_old = np.linspace(0, 100, len(cycle))
    interp_func = interp1d(x_old, cycle, kind="linear", fill_value="extrapolate")
    return interp_func(np.linspace(0, 100, target_size))


def extract_cycles_by_maximums(local_minima, local_maxima):
    """Строит циклы походки по парам соседних максимумов."""
    gait_cycles = []
    local_minima = np.asarray(local_minima)
    local_maxima = np.asarray(local_maxima)
    size = min(len(local_minima), len(local_maxima))

    for i in range(size - 1):
        max_left = local_maxima[i]
        max_right = local_maxima[i + 1]
        min_after_max_left = local_minima[local_minima > max_left]
        min_after_max_right = local_minima[local_minima > max_right]
        if len(min_after_max_left) > 0 and len(min_after_max_right) > 0:
            gait_cycles.append({"min_left": int(min_after_max_left[0]), "min_right": int(min_after_max_right[0])})
    return gait_cycles


def filter_cycles(cycles, min_frames=50, max_frames=80):
    """Фильтрует циклы по длине."""
    return [(phase["min_left"], phase["min_right"]) for phase in cycles
            if min_frames <= (phase["min_right"] - phase["min_left"]) <= max_frames]



In [72]:


def extract_cycle_by_knees(
    df_list,
    # параметры для левой ноги
    left_y_min_min=0,   left_y_min_max=5,
    left_y_max_min=55,  left_y_max_max=80,
    left_min_frames=60, left_max_frames=85,
    left_shift_frames=0,
    # параметры для правой ноги
    right_y_min_min=-30,   right_y_min_max=5,
    right_y_max_min=55,  right_y_max_max=80,
    right_min_frames=60, right_max_frames=85,
    right_shift_frames=0,
):
    """Выделяет циклы походки по углу в колене."""
    knee_cycles = []

    leg_configs = {
        "lknee-fle-ext": dict(
            y_min_min=left_y_min_min,   y_min_max=left_y_min_max,
            y_max_min=left_y_max_min,   y_max_max=left_y_max_max,
            min_frames=left_min_frames, max_frames=left_max_frames,
            shift_frames=left_shift_frames,
        ),
        "rknee-fle-ext": dict(
            y_min_min=right_y_min_min,   y_min_max=right_y_min_max,
            y_max_min=right_y_max_min,   y_max_max=right_y_max_max,
            min_frames=right_min_frames, max_frames=right_max_frames,
            shift_frames=right_shift_frames,
        ),
    }

    for df in df_list:
        n_frames = len(df)

        for col, cfg in leg_configs.items():
            if col not in df.columns:
                knee_cycles.append([])
                continue

            y = df[col].astype(float).values
            peaks_min, _ = find_peaks(-y)
            peaks_max, _ = find_peaks(y)

            peaks_min_vals = y[peaks_min]
            peaks_max_vals = y[peaks_max]

            filtered_peaks_min = peaks_min[
                (peaks_min_vals >= cfg["y_min_min"]) & (peaks_min_vals <= cfg["y_min_max"])
            ]
            filtered_peaks_max = peaks_max[
                (peaks_max_vals >= cfg["y_max_min"]) & (peaks_max_vals <= cfg["y_max_max"])
            ]

            cycles = extract_cycles_by_maximums(filtered_peaks_min, filtered_peaks_max)
            cycles = filter_cycles(cycles, min_frames=cfg["min_frames"], max_frames=cfg["max_frames"])

            shift = cfg["shift_frames"]
            if shift != 0:
                shifted = []
                for (L, R) in cycles:
                    new_L = max(0, L - shift)
                    new_R = max(new_L + 1, min(n_frames, R - shift))
                    shifted.append((new_L, new_R))
                cycles = shifted

            knee_cycles.append(cycles)

    return knee_cycles




### Алгоритмы для усреднения данных



In [73]:

def to_one_size(cycles, target_size=100):
    """Приводит список циклов к одной длине."""
    return np.array([interpolate_cycle(c, target_size=target_size) for c in cycles])

def cycle_averaging(df_list, target_size=100, **extract_kwargs):
    """Усреднение циклов походки по всем записям."""
    knee_cycles = extract_cycle_by_knees(df_list, **extract_kwargs)
    average_cycles = [[] for _ in range(6)]

    for k in range(len(knee_cycles)):
        cycles_data = knee_cycles[k]
        rec_idx = k // 2
        df = df_list[rec_idx]
        for (L, R) in cycles_data:
            for seg_idx in range(6):
                col = FLE_EXT_COLS[seg_idx]
                if col not in df.columns:
                    continue
                if k % 2 == 0 and seg_idx in (0, 2, 4):
                    average_cycles[seg_idx].append(df[col].astype(float).values[L:R])
                elif k % 2 == 1 and seg_idx in (1, 3, 5):
                    average_cycles[seg_idx].append(df[col].astype(float).values[L:R])

    fully_average_cycles = []
    for i in range(6):
        if average_cycles[i]:
            interpolated = to_one_size(average_cycles[i], target_size=target_size)
            mean_cycle = np.mean(interpolated, axis=0)
            fully_average_cycles.append(mean_cycle - mean_cycle[0])
        else:
            fully_average_cycles.append(np.full(target_size, np.nan))
    return fully_average_cycles


def cycle_averaging_for_patient(patients_data, patient_name, target_size=100, **extract_kwargs):
    """Усреднённые циклы походки для одного пациента."""
    if patient_name not in patients_data:
        return [np.full(target_size, np.nan)] * 6
    return cycle_averaging(
        patients_data[patient_name],
        target_size=target_size,
        **extract_kwargs,
    )

def get_amputation_side(patient_name):
    """Сторона ампутации из имени пациента."""
    s = (patient_name or "").strip()
    if s.startswith("П "):
        return "П"
    if s.startswith("Л "):
        return "Л"
    return None




### Файл конфига для каждого пациента

In [74]:
shift_config = {
    "ottobock cleg": {
        "left_y_min_min": 0,   "left_y_min_max": 5,
        "left_y_max_min": 55,  "left_y_max_max": 80,
        "left_min_frames": 60, "left_max_frames": 85,
        "left_shift_frames": -3,
        "right_y_min_min": -30,  "right_y_min_max": 5,
        "right_y_max_min": 55,  "right_y_max_max": 80,
        "right_min_frames": 60, "right_max_frames": 85,
        "right_shift_frames": 0,
    },

    "Актив-2": {
        "left_y_min_min": 0,   "left_y_min_max": 5,
        "left_y_max_min": 55,  "left_y_max_max": 80,
        "left_min_frames": 60, "left_max_frames": 85,
        "left_shift_frames": 7,
        "right_y_min_min": -30,  "right_y_min_max": 5,
        "right_y_max_min": 55,  "right_y_max_max": 80,
        "right_min_frames": 60, "right_max_frames": 85,
        "right_shift_frames": 0,
    },
}


### Сохранение графиков в excel

In [75]:
def export_cycles_to_excel_from_plot_logic(
    patients_data,
    output_dir="cycles_export",
    patient_names=None,
    target_size=100,
    shift_config=None,
):
    os.makedirs(output_dir, exist_ok=True)

    if patient_names is None:
        patient_names = list(patients_data.keys())
    else:
        patient_names = [n for n in patient_names if n in patients_data]

    if shift_config is None:
        shift_config = {}

    x = np.linspace(0, 100, target_size)

    SHEET_NAMES = [
        "Левое бедро",
        "Правое бедро",
        "Левое колено",
        "Правое колено",
        "Левый голеностоп",
        "Правый голеностоп",
    ]

    for pname in patient_names:

        print(f"Обработка: {pname}")

        df_list = patients_data[pname]
        cfg = shift_config.get(pname, {})

        knee_cycles = extract_cycle_by_knees(df_list, **cfg)

        # Полностью повторяем логику plot_all_cycles_overlay
        per_segment = [[] for _ in range(6)]

        for k in range(len(knee_cycles)):

            cycles_data = knee_cycles[k]

            rec_idx = k // 2
            df = df_list[rec_idx]

            for (L, R) in cycles_data:

                for seg_idx in range(6):

                    col = FLE_EXT_COLS[seg_idx]

                    if col not in df.columns:
                        continue

                    if k % 2 == 0 and seg_idx in (0, 2, 4):

                        raw = df[col].astype(float).values[L:R]

                        cycle = interpolate_cycle(
                            raw,
                            target_size
                        )

                        cycle = cycle - cycle[0]

                        per_segment[seg_idx].append(cycle)

                    elif k % 2 == 1 and seg_idx in (1, 3, 5):

                        raw = df[col].astype(float).values[L:R]

                        cycle = interpolate_cycle(
                            raw,
                            target_size
                        )

                        cycle = cycle - cycle[0]

                        per_segment[seg_idx].append(cycle)

        safe_name = "".join(
            c if c not in r'\/:*?"<>|' else "_"
            for c in pname
        )

        filepath = os.path.join(
            output_dir,
            f"{safe_name}.xlsx"
        )

        with pd.ExcelWriter(filepath, engine="openpyxl") as writer:

            for seg_idx, sheet_name in enumerate(SHEET_NAMES):

                data = {
                    "Цикл (%)": x
                }

                for step_idx, cycle in enumerate(
                    per_segment[seg_idx],
                    start=1
                ):
                    data[f"Шаг {step_idx}"] = cycle

                df_export = pd.DataFrame(data)

                df_export.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    index=False
                )

        print(f"✅ Сохранён: {filepath}")

    print(
        f"\nГотово.\nПапка: {os.path.abspath(output_dir)}"
    )

In [76]:
# export_cycles_to_excel_from_plot_logic(
#     patients_data_tta,
#     output_dir="cycles_export",
#     shift_config=shift_config,
# )

### Красивая отрисовка всех графиков для отчёта (каждый шаг + усредненный цикл)

In [77]:
def extract_all_cycles(
    patients_data,
    patient_name,
    target_size=100,
    shift_config=None,
):
    if shift_config is None:
        shift_config = {}

    df_list = patients_data[patient_name]

    cfg = shift_config.get(patient_name, {})
    knee_cycles = extract_cycle_by_knees(df_list, **cfg)

    per_segment = [[] for _ in range(6)]

    for k in range(len(knee_cycles)):

        cycles_data = knee_cycles[k]

        rec_idx = k // 2
        df = df_list[rec_idx]

        for (L, R) in cycles_data:

            for seg_idx in range(6):

                col = FLE_EXT_COLS[seg_idx]

                if col not in df.columns:
                    continue

                if k % 2 == 0 and seg_idx in (0, 2, 4):

                    raw = df[col].astype(float).values[L:R]

                    cycle = interpolate_cycle(raw, target_size)

                    per_segment[seg_idx].append(cycle)

                elif k % 2 == 1 and seg_idx in (1, 3, 5):

                    raw = df[col].astype(float).values[L:R]

                    cycle = interpolate_cycle(raw, target_size)

                    per_segment[seg_idx].append(cycle)

    return per_segment

In [78]:
def prepare_cycles_for_plot(
    patients_data,
    patient_names=None,
    target_size=100,
    shift_config=None,
    amp_side="П"
):

    if patient_names is None:
        patient_names = list(patients_data.keys())
    else:
        patient_names = [n for n in patient_names if n in patients_data]

    if shift_config is None:
        shift_config = {}

    all_patients_cycles = {}

    for pname in patient_names:

        df_list = patients_data[pname]

        # amp_side = get_amputation_side(pname)

        cfg = shift_config.get(pname, {})
        knee_cycles = extract_cycle_by_knees(df_list, **cfg)

        per_segment = [[] for _ in range(6)]

        for k in range(len(knee_cycles)):

            cycles_data = knee_cycles[k]

            rec_idx = k // 2
            df = df_list[rec_idx]

            for (L, R) in cycles_data:

                for seg_idx in range(6):

                    col = FLE_EXT_COLS[seg_idx]

                    if col not in df.columns:
                        continue

                    if k % 2 == 0 and seg_idx in (0, 2, 4):

                        raw = df[col].astype(float).values[L:R]

                        per_segment[seg_idx].append(
                            interpolate_cycle(raw, target_size)
                        )

                    elif k % 2 == 1 and seg_idx in (1, 3, 5):

                        raw = df[col].astype(float).values[L:R]

                        per_segment[seg_idx].append(
                            interpolate_cycle(raw, target_size)
                        )

        if amp_side == "П":

            ordered = [
                per_segment[1],
                per_segment[3],
                per_segment[5],
                per_segment[0],
                per_segment[2],
                per_segment[4],
            ]

        elif amp_side == "Л":

            ordered = [
                per_segment[0],
                per_segment[2],
                per_segment[4],
                per_segment[1],
                per_segment[3],
                per_segment[5],
            ]

        else:
            ordered = per_segment

        all_patients_cycles[pname] = ordered

    return all_patients_cycles

In [79]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np


def plot_cycles_report(
    all_patients_cycles,
    target_size=100,
    title="Кинематика ходьбы",
    alpha=0.25,
):

    segment_names = [
        "Бедро",
        "Бедро",
        "Колено",
        "Колено",
        "Голеностоп",
        "Голеностоп",
    ]

    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=segment_names,
        vertical_spacing=0.08,
        horizontal_spacing=0.08,
    )

    fig.add_annotation(
    text="<b>Протезированная конечность</b>",
    x=0.1,
    y=1.06,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=18),
    )

    fig.add_annotation(
        text="<b>Интактная конечность</b>",
        x=0.87,
        y=1.06,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=18),
    )

    x = np.linspace(0, 100, target_size)

    colors = [
        "#1f77b4",
        "#d62728",
        "#2ca02c",
        "#ff7f0e",
        "#9467bd",
        "#17becf",
        "#e377c2",
        "#8c564b",
    ]

    patient_colors = {
        pname: colors[i % len(colors)]
        for i, pname in enumerate(all_patients_cycles.keys())
    }

    legend_shown = set()

    for pname, ordered in all_patients_cycles.items():

        color = patient_colors[pname]

        for seg_idx in range(6):

            row = (seg_idx % 3) + 1
            col = (seg_idx // 3) + 1

            curves = []

            for step_i, y in enumerate(ordered[seg_idx]):

                y_norm = y - y[0]

                curves.append(y_norm)

                show_legend = (
                    pname not in legend_shown
                    and seg_idx == 0
                    and step_i == 0
                )

                if show_legend:
                    legend_shown.add(pname)

                fig.add_trace(
                    go.Scatter(
                        x=x,
                        y=y_norm,
                        mode="lines",
                        line=dict(
                            color=color,
                            width=1,
                        ),
                        opacity=alpha,
                        name=pname,
                        legendgroup=pname,
                        showlegend=show_legend,
                        hovertemplate=(
                            f"{pname}<br>"
                            f"Шаг {step_i}<br>"
                            "% цикла: %{x:.1f}<br>"
                            "Угол: %{y:.2f}°"
                            "<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )

            if len(curves):

                mean_curve = np.mean(
                    np.vstack(curves),
                    axis=0,
                )

                fig.add_trace(
                    go.Scatter(
                        x=x,
                        y=mean_curve,
                        mode="lines",
                        line=dict(
                            color=color,
                            width=4,
                        ),
                        showlegend=False,
                        hovertemplate=(
                            f"{pname} (среднее)<br>"
                            "% цикла: %{x:.1f}<br>"
                            "Угол: %{y:.2f}°"
                            "<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )

    fig.update_layout(
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1300,
        height=1500,
        title=title,
        font=dict(size=14),
        legend=dict(
            orientation="h",
            y=-0.05,
            x=0.5,
            xanchor="center",
        ),
    )

    fig.update_xaxes(
        title_text="Цикл походки, %",
        showgrid=True,
        gridcolor="lightgray",
    )

    fig.update_yaxes(
        title_text="Угол сгибания, °",
        showgrid=True,
        gridcolor="lightgray",
    )

    return fig

In [80]:
all_patients_cycles = prepare_cycles_for_plot(
    patients_data_tta,
    shift_config=shift_config,
)

In [81]:
fig = plot_cycles_report(
    all_patients_cycles,
    title="",
)

fig.show()

### Удаление выбросов из всех шагов

In [82]:
def remove_steps(
    all_patients_cycles,
    patient_name,
    segment_idx,
    bad_steps,
):

    all_patients_cycles[patient_name][segment_idx] = [

        step

        for i, step in enumerate(
            all_patients_cycles[patient_name][segment_idx]
        )

        if i not in bad_steps
    ]

In [83]:
# remove_steps(
#     all_patients_cycles,
#     patient_name="Ottobock cleg",
#     segment_idx=3,
#     bad_steps=[5]
# )

# remove_steps(
#     all_patients_cycles,
#     patient_name="Ottobock cleg",
#     segment_idx=5,
#     bad_steps=[33]


In [84]:
fig_cleg_vs_activ_steps = plot_cycles_report(
    all_patients_cycles,
    title="",
)

fig_cleg_vs_activ_steps.show()

### Отрисовка сырых данных для отдельного сустава

In [85]:

from scipy.signal import find_peaks


def detect_main_peaks(
    y,
    prominence=50,
    min_distance=20
):
    """
    Поиск основных максимумов по prominence.
    """

    peaks, props = find_peaks(
        y,
        prominence=prominence,
        distance=min_distance
    )

    return peaks



In [86]:

def peaks_with_gaps(t, y, peaks, max_gap=100):
    """
    Добавляет разрывы между удалёнными максимумами.
    max_gap - максимальное допустимое расстояние между пиками.
    """

    x_plot = []
    y_plot = []

    for i in range(len(peaks)):

        x_plot.append(t[peaks[i]])
        y_plot.append(y[peaks[i]])

        if i < len(peaks) - 1:

            gap = peaks[i + 1] - peaks[i]

            if gap > max_gap:
                x_plot.append(np.nan)
                y_plot.append(np.nan)

    return np.array(x_plot), np.array(y_plot)


In [87]:
def get_cycle_amplitudes(y, peaks):
    """
    Для каждого цикла между соседними максимумами ищет минимум
    и вычисляет амплитуду.

    Возвращает список словарей:
    {
        peak_idx,
        peak_value,
        valley_idx,
        valley_value,
        amplitude
    }
    """

    cycles = []

    for i in range(len(peaks) - 1):

        start = peaks[i]
        end = peaks[i + 1]

        cycle = y[start:end]

        if len(cycle) == 0:
            continue

        valley_local = np.argmin(cycle)
        valley_idx = start + valley_local

        peak_value = y[start]
        valley_value = y[valley_idx]

        cycles.append({
            "peak_idx": start,
            "peak_value": peak_value,
            "valley_idx": valley_idx,
            "valley_value": valley_value,
            "amplitude": peak_value - valley_value
        })

    return cycles

In [88]:
def plot_joint_for_patient(
    patients_data,
    patient_name,
    joint_idx,
):

    recordings = patients_data[patient_name]

    all_columns = []
    for cols in LIMB_COLUMNS.values():
        all_columns.extend(cols)

    if joint_idx >= len(all_columns):
        raise ValueError(
            f"joint_idx должен быть от 0 до {len(all_columns)-1}"
        )

    target_col = all_columns[joint_idx]

    n_records = len(recordings)

    fig = make_subplots(
        rows=n_records,
        cols=1,
        shared_xaxes=False,
        subplot_titles=[
            f"Запись {i+1}"
            for i in range(n_records)
        ],
        vertical_spacing=0.04
    )

    for row, df in enumerate(recordings, start=1):

        if target_col not in df.columns:
            continue

        y = df[target_col].astype(float).values
        t = np.arange(len(y))

        # исходный сигнал
        fig.add_trace(
            go.Scatter(
                x=t,
                y=y,
                mode="lines",
                name=f"Запись {row}",
                showlegend=False
            ),
            row=row,
            col=1
        )

        # основные максимумы
        peaks = detect_main_peaks(
            y,
            prominence=50,
            min_distance=20
        )

        # красная огибающая
        x_env, y_env = peaks_with_gaps(
            t,
            y,
            peaks,
            max_gap=100
        )

        fig.add_trace(
            go.Scatter(
                x=x_env,
                y=y_env,
                mode="lines+markers",
                line=dict(
                    color="red",
                    width=3
                ),
                showlegend=False
            ),
            row=row,
            col=1
        )

        # расчет амплитуд
        cycles = get_cycle_amplitudes(
            y,
            peaks
        )

        for cycle in cycles:

            peak_idx = cycle["peak_idx"]
            valley_idx = cycle["valley_idx"]

            peak_value = cycle["peak_value"]
            valley_value = cycle["valley_value"]

            amplitude = cycle["amplitude"]

            # максимум
            fig.add_trace(
                go.Scatter(
                    x=[peak_idx],
                    y=[peak_value],
                    mode="markers",
                    marker=dict(
                        size=5,
                        color="red"
                    ),
                    showlegend=False
                ),
                row=row,
                col=1
            )

            # минимум
            fig.add_trace(
                go.Scatter(
                    x=[valley_idx],
                    y=[valley_value],
                    mode="markers",
                    marker=dict(
                        size=5,
                        color="blue"
                    ),
                    showlegend=False
                ),
                row=row,
                col=1
            )

            # вертикальная линия амплитуды
            fig.add_trace(
                go.Scatter(
                    x=[peak_idx, peak_idx],
                    y=[peak_value, valley_value],
                    mode="lines",
                    line=dict(
                        color="purple",
                        width=1,
                        dash="dash"
                    ),
                    showlegend=False,
                    hovertemplate=
                    f"Амплитуда: {amplitude:.1f}°"
                    "<extra></extra>"
                ),
                row=row,
                col=1
            )

            # подпись амплитуды
            fig.add_annotation(
                x=peak_idx,
                y=-7,
                text=f"{amplitude:.1f}°",
                showarrow=False,
                font=dict(size=10),
                row=row,
                col=1
            )

        fig.update_yaxes(
            title_text="Угол сгибания, °",
            row=row,
            col=1
        )

    fig.update_xaxes(
        title_text="Номер кадра записи",
        row=n_records,
        col=1
    )

    fig.update_layout(
        height=300 * n_records,
        title_text=f"Протез: {patient_name}",
        margin=dict(t=80, b=40),
        showlegend=False
    )

    return fig

In [89]:

patient_name = list(patients_data_tta.keys())[0]

fig_cleg_knee = plot_joint_for_patient(
    patients_data_tta,
    patient_name,
    joint_idx=2
)

fig_cleg_knee.show()



In [90]:

patient_name = list(patients_data_tta.keys())[1]

fig_aktiv_knee = plot_joint_for_patient(
    patients_data_tta,
    patient_name,
    joint_idx=2
)

fig_aktiv_knee.show()




### Статистика по углам сгибания коленных модулей



In [91]:


name_cleg = list(patients_data_tta.keys())[0]
name_activ = list(patients_data_tta.keys())[1]

def amplitude_statistics(cycles):

    if len(cycles) == 0:
        return None

    amplitudes = np.array([
        cycle["amplitude"]
        for cycle in cycles
    ])

    return {
        "Количество шагов": len(amplitudes),

        "Средняя амплитуда сгибания": np.mean(amplitudes),

        "Медианная амплитуда сгибания": np.median(amplitudes),

        "Стандартное отклонение": np.std(amplitudes),

        "Коэффициент вариации (%)": (
            np.std(amplitudes) / np.mean(amplitudes) * 100
            if np.mean(amplitudes) != 0
            else np.nan
        ),

        "Минимальная амплитуда сгибания": np.min(amplitudes),

        "Максимальная амплитуда сгибания": np.max(amplitudes),

        "Размах амплитуд": (
            np.max(amplitudes)
            - np.min(amplitudes)
        ),
    }

In [92]:
def build_patient_peak_table(
    patients_data,
    patient_name,
    joint_idx,
    prominence=50,
    min_distance=20
):

    recordings = patients_data[patient_name]

    rows = []

    all_columns = []
    for cols in LIMB_COLUMNS.values():
        all_columns.extend(cols)

    target_col = all_columns[joint_idx]

    for rec_idx, df in enumerate(recordings):

        y = df[target_col].astype(float).values

        peaks = detect_main_peaks(
            y,
            prominence=prominence,
            min_distance=min_distance
        )

        cycles = get_cycle_amplitudes(
            y,
            peaks
        )

        stats = amplitude_statistics(
            cycles
        )

        if stats is None:
            continue

        row = {
            "Номер записи": rec_idx + 1,
            **stats
        }

        rows.append(row)

    return pd.DataFrame(rows)


statistics_table_cleg = build_patient_peak_table(
    patients_data_tta,
    patient_name=name_cleg,
    joint_idx=2
)

display(statistics_table_cleg)
statistics_table_activ = build_patient_peak_table(
    patients_data_tta,
    patient_name=name_activ,
    joint_idx=2
)

display(statistics_table_activ)

,Номер записи,Количество шагов,Средняя амплитуда сгибания,Медианная амплитуда сгибания,Стандартное отклонение,Коэффициент вариации (%),Минимальная амплитуда сгибания,Максимальная амплитуда сгибания,Размах амплитуд
0,1,33,64.887362,64.947760,3.230220,4.978196,53.17138,72.671198,19.499818
1,2,30,63.153150,64.025165,4.532931,7.177679,50.32820,69.562680,19.234480


,Номер записи,Количество шагов,Средняя амплитуда сгибания,Медианная амплитуда сгибания,Стандартное отклонение,Коэффициент вариации (%),Минимальная амплитуда сгибания,Максимальная амплитуда сгибания,Размах амплитуд
0,1,39,66.258202,66.70221,3.789612,5.719461,50.26958,71.81363,21.54405


### Формирование html отчёта со всеми графиками


In [93]:
from plotly.io import to_html

graph_html_cleg_knee = to_html(
    fig_cleg_knee,
    full_html=False,
    include_plotlyjs="cdn"
)

graph_html_aktiv_knee = to_html(
    fig_aktiv_knee,
    full_html=False,
    include_plotlyjs="cdn"
)

graph_html_cleg_vs_activ_steps = to_html(
    fig_cleg_vs_activ_steps,
    full_html=False,
    include_plotlyjs="cdn"
)

statistics_table_html_cleg = statistics_table_cleg.round(2).to_html(
    index=False,
    classes="stats-table",
    border=1
)

statistics_table_html_activ = statistics_table_activ.round(2).to_html(
    index=False,
    classes="stats-table",
    border=1
)

In [94]:
html = f"""
<style>
.stats-table {{
    margin-left: auto;
    margin-right: auto;
    border-collapse: collapse;
}}

.stats-table th,
.stats-table td {{
    text-align: center;
    padding: 6px 12px;
}}

.stats-table th {{
    background-color: #f0f0f0;
}}
</style>
<html>
<head>
<meta charset="utf-8">
<title>Отчёт</title>
</head>

<body>

<h1>Сравнительный анализ кинематики ходьбы на коленных модулях Ottobock cleg и Актив-2</h1>

<p>Уважаемые коллеги, предлагаем вам ознакомиться с результатами анализа кинематики ходьбы Макаровой Ярославы на двух коленных модулях: Ottobock cleg и Актив-2. </p>

<p>В данном документе все графики интерактивные. Это означает, что вы можете:
<ol>
    <li>Менять масштаб графиков с помощью мыши;</li>
    <li>Подробно рассмотреть углы сгибания суставов на различных этапах цикла ходьбы;</li>
    <li>Сохранять нужные фрагменты графиков в виде изображений для последующего анализа и включения в отчёты.</li>
</ol>
</p>

<p>
По всем вопросам обработки данных и интерпретации результатов можете обращаться к Дарье в телеграм или в макс:
<ol>
    <li><a href="https://t.me/korostovskayadarya">Телеграм</a>  </li>
    <li><a href="https://max.ru/u/f9LHodD0cOI-ut55JRCu8BzeJep7vHFrm8nhtmfYieMOVAloqBLKmNR9QvA">Макс</a>  </li>
</ol>
</p>

<p> В этом документе все данные были получены с помощью <b>коммерчески доступной инерциальной системы захвата движений Perception Neuron 3.</b>
</p>


<h2>1. Анализ исходных данных</h2>
<p>В данном разделе представлены исходные данные углов сгибания коленных модулей во время ходьбы. Всего было снято по 4 записи ходьбы на каждом коленном модуле. Была отрисована красная линия "стабильности" максимального угла сгибания коленного модуля, чтобы можно было визуально оценить, насколько сильный есть разброс этой величины. 
</p>

<p>Также были рассчитаны статистические показатели для максимальных углов сгибания.<b> Все показатели в статистических таблицах приведены в градусах </b> за исключением коэффициента вариации.</p>

<h3>1.1 Необработанные записи ходьбы для Ottobock cleg</h3>
{graph_html_cleg_knee}

<h4>Статистическая сводка для углов сгибания Ottobock cleg</h3>
{statistics_table_html_cleg}

<p>Средняя амплитуда сгибания различается не более чем на 1 градус. Стандартное отклонение не превосходит 2.5 градусов. Размах амплитуд не превосходит 10 градусов.</p>

<h3>1.2 Необработанные записи ходьбы для Актив-2</h3>
{graph_html_aktiv_knee}

<h4>Статистическая сводка для углов сгибания Актив-2</h3>
{statistics_table_html_activ}

<p>Средняя амплитуда сгибания различается не более чем на 4 градуса. Стандартное отклонение не превосходит 4 градусов. Размах амплитуд не превосходит 17.5 градусов, но она <b> не меньше 10 градусов </b> в каждой записи.</p>

<h3>1.3 Выводы</h3>
<p>Ottobock cleg демонститует небольшую разницу между средними амплитудами. Сгибание коленного модуля достаточно стабильное, поскольку стандартное отклонение между амплитудами сгибания и размах амплитуд достаточно небольшие. Коленный модуль способен обеспечить стабильную ходьбу для пациента.</p>

<p>Актив-2 демонститует небольшую нестабильность в работе коленного модуля. Стандартное отклонение между амплитудами сгибания в 2 раза выше чем у ottobock cleg. Также наблюдается большая разница между размахом амплитуд. Коленный модуль способен обеспечить комфортную ходьбу для пациента, однако требуется небольшая доработка для повышения стабильности работы модуля.</p>

<h2>2. Анализ обработанных данных</h2>
<p>На графике представлены углы сгибания бедра, колена и голеностопа во время ходьбы на обоих протезах. Красным цветов обозначены данные Актив-2, синим - ottobock cleg.</p>

<p>Из исходных данных были выделены циклы походки - шаги. В данном случае 1 шаг - это момент от удара пятки до следующего удара пятки для одной ноги.</p>

<p>Здесь представлены как отдельные шаги, так и усредненные циклы походки, которые обозначены жирными линиями. Если вы хотите рассмотреть данные только для одного коленного модуля, то в легенде графика (в самом низу) вы можете кликнуть на название коленного модуля, который вы хотите скрыть./p>

<h3>2.1. Данные углов сгибания бедра, колена и голеностопа</h3>

{graph_html_cleg_vs_activ_steps}

<h3>2.2 Выводы</h3>

<p>Усредненные углы сгибания <b>бедра у протезированной конечности</b> на обоих коленных модулях имеют различия до 3 градусов. Все значения входят в интервал +-10 градусов от усредненных значений.<br>

Усредненные углы сгибания <b>берда интактной конечности</b> имеют различия до 6 градусов. Все значения входят в интервал +-5 градусов от усредненных значений</p>

<p>Усредненные углы сгибания <b>колена у протезированной конечности</b> на обоих коленных модулях имеют различия до 4 градусов. Наблюдается большой разброс между кинематикой шагов для обоих коленных модулей.<br>

Усредненные углы сгибания <b>колена интактной конечности</b> имеют различия до 2 градусов. Все значения входят в интервал +-5 градусов от усредненных значений</p>


<p>Усредненные углы сгибания <b>голеностопа у протезированной конечности</b> на обоих коленных модулях имеют различия до 10 градусов. Это обусловлено тем, что были на протезах разные протезы стоп - ottobock taleo (модуль cleg) и ottobock trias (модуль актив-2). Данный график демонстрирует различие кинематики взятых протезов стоп из-за их формы и профиля, а не из-за используемых коленных модулей.<br>

Усредненные углы сгибания <b>голеностопа интактной конечности</b> имеют различия до 2 градусов. Все значения входят в интервал +-5 градусов от усредненных значений</p>


<h2>Заключение</h2>

<p>Статистический анализ амплитуд сгибания коленных модулей выявил небольшую нестабильность в работе модуля Актив-2.<br>
Несмотря на это, кинематика интактной конечности практически не зависит от коленного модуля. Это говорит о том, что пациент способен компенсировать нестабильность работы коленного модуля и подстраиваться под протез.
</p>

</body>
</html>
"""

In [95]:
with open("report_PN3_system.html", "w", encoding="utf-8") as f:
    f.write(html)